NOTE: These results are stimulated as we don't have live data of user behaviour after intervetion.

# Business scenario question:

Can a targeted retention intervention can reduce the churn among those high risk customers?

From SHAP analysis we know contract month-to-month feature is prone to churn more. Goal is to identify which customers whould likely to change their behaviour because of retention intervention. 

# Intervention: 
Offer retention incentive to selected high risk customers

- personalized discount
- offer services with discounted price
- free security add-on trial
- contract upgrade incentive 
- loyalty reward

# Primary outcome

Customer churn after intervention

# Business Objectives

1. Did the intervention reduced churn?
2. Whats the avg treatment effect (ATE)?
3. which customer segment responded strongly?
4. Incremental revenue
5. ROI


In [19]:
#loading data
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "code").is_dir() and (path / "model").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing code/ and model/")

DATA_DIR = PROJECT_ROOT / "code" / "data"
MODEL_DIR = PROJECT_ROOT / "model"
df = pd.read_csv(DATA_DIR / "Edited_Data.csv")

In [20]:
df.shape
df.columns.tolist()

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'PaperlessBilling',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'avg_monthly_charge',
 'charge_ratio',
 'service_score',
 'MultipleLines_No',
 'MultipleLines_No phone service',
 'MultipleLines_Yes',
 'InternetService_DSL',
 'InternetService_Fiber optic',
 'InternetService_No',
 'OnlineSecurity_No',
 'OnlineSecurity_No internet service',
 'OnlineSecurity_Yes',
 'OnlineBackup_No',
 'OnlineBackup_No internet service',
 'OnlineBackup_Yes',
 'DeviceProtection_No',
 'DeviceProtection_No internet service',
 'DeviceProtection_Yes',
 'TechSupport_No',
 'TechSupport_No internet service',
 'TechSupport_Yes',
 'StreamingTV_No',
 'StreamingTV_No internet service',
 'StreamingTV_Yes',
 'StreamingMovies_No',
 'StreamingMovies_No internet service',
 'StreamingMovies_Yes',
 'Contract_Month-to-month',
 'Contract_One year',
 'Contract_Two year',
 'PaymentMethod_Bank transfer (automatic)',
 'PaymentMethod_Credit card (

In [21]:
best_xgb  = joblib.load(MODEL_DIR / "xgb_best_model.pkl")
scaler    = joblib.load(MODEL_DIR / "scaler.pkl")
feat_names = joblib.load(MODEL_DIR / "feature_names.pkl")


In [22]:
causal_df = df.copy()
X = causal_df.loc[:, feat_names]
X_scaled_df = pd.DataFrame(scaler.transform(X), columns=feat_names)
causal_df['churn_probability'] = best_xgb.predict_proba(X_scaled_df)[:, 1]
causal_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_0-1yr,tenure_group_1-2yr,tenure_group_2-4yr,tenure_group_4-5yr,tenure_group_5-6yr,churn_probability
0,0,0,1,0,1,0,1,29.85,29.85,0,...,False,False,True,False,True,False,False,False,False,0.687027
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,False,False,False,True,False,False,True,False,False,0.040650
2,1,0,0,0,2,1,1,53.85,108.15,1,...,False,False,False,True,True,False,False,False,False,0.319547
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,True,False,False,False,False,False,True,False,False,0.038350
4,0,0,0,0,2,1,1,70.70,151.65,1,...,False,False,True,False,True,False,False,False,False,0.641818


In [23]:
# Target customers : customers with high churn probability and have month-month contract

#60% high risk as mentioned in the previous analysis
target_customers = causal_df[(causal_df['churn_probability']>= 0.60) 
                             & (causal_df['Contract_Month-to-month'] == 1)]


print("total target_customers", len(target_customers))
target_customers[['churn_probability', 'Contract_Month-to-month', 'tenure']]



total target_customers 911


,churn_probability,Contract_Month-to-month,tenure
0,0.687027,True,1
4,0.641818,True,2
5,0.833987,True,8
20,0.750977,True,1
31,0.743761,True,2
...,...,...,...
7008,0.656343,True,3
7010,0.864184,True,1
7018,0.745047,True,1
7032,0.878112,True,1


In [24]:
#Treatment and control groups for causal inference analysis
# treatment group: who recieves the intervention
# control group: who does not receive the intervention (business as usual)

np.random.seed(42)
#randomly assign target customers to treatment and control groups
target_customers['group'] = np.random.choice(['treatment', 'control'], size=len(target_customers), p= [0.5, 0.5])
target_customers.head()
target_customers.groupby('group').size()

group
control      455
treatment    456
dtype: int64

In [25]:
# Checking randomization of treatment and control groups
# Before running experiment, checkif the randomization has resulted in balanced groups
target_customers.groupby('group').agg(
    customers = ('group','size'),
    avg_tenure = ('tenure','mean'),
    avg_monthly_charge = ('MonthlyCharges', 'mean'),
    avg_churn_prob = ('churn_probability', 'mean')
)

,customers,avg_tenure,avg_monthly_charge,avg_churn_prob
group,,,,
control,455,5.619780,79.230659,0.718881
treatment,456,5.589912,81.047478,0.729440


In [26]:
# Simulated post intervention data: churned_after_intervention
# Usually after intervention we collect the data and found the churn rate for each group, but since it doesn't exist yet, we simulate it
# suppose : 40% churn rate in the control group and 32% churn rate in the treatment group  

np.random.seed(42)

control_grp = target_customers['group'] == 'control'
treatment_grp = target_customers['group'] == 'treatment'

target_customers.loc[control_grp, 'churned_after_intervention'] = np.random.binomial(1,0.40, control_grp.sum())
target_customers.loc[treatment_grp, 'churned_after_intervention'] = np.random.binomial(1,0.32, treatment_grp.sum())
target_customers.groupby('group')['churned_after_intervention'].mean()

# Now we have simulated data which matches the expected distribution of churn rates for each group

control_churned = control['churned_after_intervention'].sum()
treatment_churned = treatment['churned_after_intervention'].sum()

NameError: name 'control' is not defined

In [ ]:
# step 1 of causal inference
# ATE: average treatment effect

#churn rate for each group
control_grp_churns = target_customers[target_customers['group'] == 'control']['churned_after_intervention'].mean()
treatment_grp_churns = target_customers[target_customers['group'] == 'treatment']['churned_after_intervention'].mean()

# calculate ATE
ate = treatment_grp_churns - control_grp_churns
print(ate)
#negative ATE indicates that the treatment reduced churn

-0.12591093117408908


In [ ]:
#Absolute reduction, relative reduction
absolute_reduction = control_grp_churns - treatment_grp_churns
relative_reduction = absolute_reduction/control_grp_churns

print("Absolute reduction:", absolute_reduction)
print("Relative reduction:", relative_reduction)

# The (simulated) intervention reduced churn by approx 12 percentage points , 
# corresponding to a relative reduction of 30% compared with the control group.


Absolute reduction: 0.12591093117408908
Relative reduction: 0.3031189083820663


In [ ]:
# Statistical significance: compare simulated group churn rates.
from statsmodels.stats.proportion import proportions_ztest

control = target_customers.loc[target_customers["group"] == "control"]
treatment = target_customers.loc[target_customers["group"] == "treatment"]

control_churn_events = int(control["churned_after_intervention"].sum())
treatment_churn_events = int(treatment["churned_after_intervention"].sum())
group_sizes = [len(control), len(treatment)]
churn_events = [control_churn_events, treatment_churn_events]
z_stat, p_value = proportions_ztest(
    count = churn_events,
    nobs = group_sizes,
    alternative = 'two-sided'
)
print('Zstatistic', z_stat)
print('p_value', p_value)

if p_value < 0.05:
    print("the difference is statistically significant")
    
else:
    print(" the difference is not statistically significant")

Zstatistic 3.9777018003007547
p_value 6.958456080982714e-05
the difference is statistically significant


In [ ]:
# That means the difference between control and treatment groups is unlikely due to random chance alone.
# There is statistically significant evidence that the retention intervention changed the churn rate compared with the control group.


In [ ]:
# Splitting the customer base by tenure bins
# Treatment effects by customer segment
target_customers['tenure_segment'] = pd.cut(
    target_customers['tenure'],
    bins=[0, 12, 24, np.inf],
    labels=['0-12 months', '13-24 months', '25+ months']
)

target_customers['tenure_segment'].value_counts()

tenure_segment
0-12 months     801
13-24 months     98
25+ months       12
Name: count, dtype: int64

### Treatment Effect by Customer Segment

Treatment effects were evaluated across customer tenure segments.

The retention intervention showed a larger simulated reduction in churn
among customers with 0–12 months and 13–24 months of tenure.

The 25+ month segment contained only a small number of customers in each
experimental group. Therefore, its estimated treatment effect is unstable
and is not used for business targeting.

This highlights the importance of considering sample size when interpreting
heterogeneous treatment effects.

In [ ]:
# Restricting to segments with a minimum sample size of 30 customers

segment_summary = (
    target_customers
    .groupby(['tenure_segment', 'group'], observed=True)
    .agg(
        customers=('churned_after_intervention', 'size'),
        churned=('churned_after_intervention', 'sum'),
        churned_after_intervention=('churned_after_intervention', 'mean')
    )
    .reset_index()
)
MIN_SAMPLE_SIZE = 30
segment_summary
segment_filtered = segment_summary['customers'] >= MIN_SAMPLE_SIZE
segment_summary_filtered = segment_summary[segment_filtered]
segment_summary_filtered

,tenure_segment,group,customers,churned,churned_after_intervention
0,0-12 months,control,398,162.0,0.407035
1,0-12 months,treatment,403,119.0,0.295285
2,13-24 months,control,52,27.0,0.519231
3,13-24 months,treatment,46,11.0,0.239130


In [ ]:
# Calulating churn rate for each tenure segment for each group

# churn_rate_segment = target_customers.groupby(['tenure_segment', 'group'])['churned_after_intervention'].mean().reset_index()
# churn_rate_segment

In [ ]:
# calculating treatment effect for each tenure segment
segment_table = segment_summary_filtered.pivot(
    index = 'tenure_segment',
    columns = 'group',
    values = 'churned_after_intervention'
)
segment_table['treatment_effect'] = segment_table['treatment'] - segment_table['control']
segment_table

group,control,treatment,treatment_effect
tenure_segment,,,
0-12 months,0.407035,0.295285,-0.11175
13-24 months,0.519231,0.239130,-0.28010


In [ ]:
# Interpreting results:
# 0-12 months the churn reduced by 11 percentage points
# 13-24 months the churn reduced by 28 percentage points
# [IMPORTANT]25+ months the churn increased by 28 percentage points as there are no churner in the control group

# It appears to be most effective in the 13-24 months tenure segment

In [ ]:
#The important transition is:

# A/B testing: Does the intervention work overall?
# Segment analysis: Where does it work?
# Uplift/CATE: Which individual customers are most likely to respond?

Object `overall` not found.
Object `work` not found.
Object `respond` not found.


In [ ]:
# CATE/ uplift modeling
# uplift modeling means estimating the causal effect of a treatment or intervention on an individual level, rather than on an average level across a population. It helps identify which individuals are most likely to respond positively to a treatment.
# Treatment effect on an individual level conditioned on customer multiple characteristics
# Business questions: Which customers are most likely to benefit from the intervention?

uplift_df = target_customers.copy()
#drop columns that are not allowed as predictors.
# user orginial tenure column instead of derived tenure_segment
# uplift model should learn: “Given the customer's characteristics, does the treatment change their outcome?”
#rather than relying on: “What did my previous churn model predict?”

columns_to_drop = ['Churn',
    'group',
    'churned_after_intervention',
    'tenure_segment',
    'churn probability'] 

X_uplift = uplift_df.drop(
    columns=columns_to_drop,
    errors='ignore'
)


In [ ]:
X_uplift.columns.to_list()
y_uplift = uplift_df['churned_after_intervention']
treatment = (
    uplift_df['group'] == 'treatment'
).astype(int)

In [ ]:
treatment

0       1
4       0
5       0
20      0
31      1
       ..
7008    0
7010    1
7018    1
7032    1
7041    0
Name: group, Length: 911, dtype: int64

In [ ]:
X_uplift.dtypes

gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                       int64
PhoneService                                 int64
PaperlessBilling                             int64
MonthlyCharges                             float64
TotalCharges                               float64
avg_monthly_charge                         float64
charge_ratio                               float64
service_score                                int64
MultipleLines_No                              bool
MultipleLines_No phone service                bool
MultipleLines_Yes                             bool
InternetService_DSL                           bool
InternetService_Fiber optic                   bool
InternetService_No                            bool
OnlineSecurity_No                             bool
OnlineSecurity_No internet serv

In [ ]:
# Using T-learner or two model approach for uplift modeling
from xgboost import XGBClassifier

# Separate treatment and control customers
control_mask = treatment == 0
treatment_mask = treatment == 1

X_control = X_uplift[control_mask]
y_control = y_uplift[control_mask]

X_treatment = X_uplift[treatment_mask]
y_treatment = y_uplift[treatment_mask]

print("Control customers:", len(X_control))
print("Treatment customers:", len(X_treatment))

#training separate models for control and treatment groups
control_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    random_state=42,
    eval_metric='logloss'
)

treatment_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    random_state=42,
    eval_metric='logloss'
)

#fitting the models
control_model.fit(X_control, y_control)
treatment_model.fit(X_treatment, y_treatment)


Control customers: 455
Treatment customers: 456


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [ ]:
# Predicting outcomes for control and treatment groups using the trained models
control_pred = control_model.predict_proba(X_uplift)[:, 1]

treatment_pred = treatment_model.predict_proba(X_uplift)[:, 1]

uplift_pred = treatment_pred - control_pred

target_customers['control_churn_probability'] = control_pred
target_customers['treatment_churn_probability'] = treatment_pred
target_customers['uplift'] = uplift_pred

In [ ]:
#Customer A: Control prediction = 0.80, Treatment prediction   = 0.50, uplift = 0.50 - 0.80 = -0.30
# The retention treatment is estimated to reduce this customer's churn probability by 30 percentage points.

# uplift = -0.02, Only a 2-point reduction → probably not worth targeting.
# uplift = +0.15, The treatment is estimated to increase churn probability → potentially a "Sleeping Dog."

target_customers[
    [
        'control_churn_probability',
        'treatment_churn_probability',
        'uplift'
    ]
].head(10)

,control_churn_probability,treatment_churn_probability,uplift
0,0.428479,0.222059,-0.206420
4,0.364492,0.229830,-0.134662
5,0.587770,0.248039,-0.339731
20,0.420330,0.297505,-0.122825
31,0.392021,0.170888,-0.221133
36,0.488986,0.217629,-0.271357
39,0.528436,0.247038,-0.281399
47,0.493581,0.320682,-0.172899
53,0.264560,0.501476,0.236916
55,0.267846,0.183793,-0.084053


In [ ]:
print(target_customers['uplift'].describe())

count    911.000000
mean      -0.117467
std        0.182304
min       -0.650982
25%       -0.231975
50%       -0.136495
75%       -0.012801
max        0.565505
Name: uplift, dtype: float64


In [ ]:
# identify the customers with the highest estimated uplift.
# more negative uplift indicates customers who are less likely to churn with the treatment.
# Ranking customers from the most treatment responsive to the least responsive.

uplift_ranking = target_customers.sort_values(by = 'uplift')[[ 'uplift',
        'control_churn_probability',
        'treatment_churn_probability',
        'tenure',
        'MonthlyCharges',
        'Contract_Month-to-month']]

uplift_ranking.head(10)

,uplift,control_churn_probability,treatment_churn_probability,tenure,MonthlyCharges,Contract_Month-to-month
5072,-0.650982,0.739844,0.088862,23,104.45,True
3427,-0.562821,0.661403,0.098581,21,94.30,True
171,-0.556271,0.740407,0.184136,2,104.40,True
2960,-0.527893,0.705216,0.177323,22,100.75,True
5242,-0.522368,0.620697,0.098329,20,88.70,True
3364,-0.504649,0.655187,0.150539,21,103.90,True
3349,-0.502035,0.636500,0.134465,11,99.50,True
6861,-0.494632,0.646851,0.152220,14,91.65,True
2339,-0.494264,0.646781,0.152517,21,94.25,True
1485,-0.492716,0.676509,0.183793,22,96.70,True


In [ ]:
# target customers with the highest estimated uplift (most likely to benefit from the treatment)
# Top 20% customers with the highest estimated uplift 
target_customers['uplift_percentile'] = target_customers['uplift'].rank(pct=True, ascending=True) #Ascending true because we are looking at negative values

#recommeded customers to target for the retention treatment
target_customers['top_20_percent'] = target_customers['uplift_percentile'] <= 0.2


In [ ]:
target_customers['top_20_percent'].value_counts()
#These 182 customers are not simply the 182 highest-churn-risk customers.
# They are the customers whose model predicts the largest benefit from the retention intervention.

top_20_percent
False    729
True     182
Name: count, dtype: int64

In [ ]:
#does uplift targeting look better than simply targeting the highest-risk customers?


#risk-based targeting group using the same 20% budget.
target_customers['risk_percentile'] = target_customers['churn_probability'].rank(pct = True, ascending=False) #Descending because higher churn probability indicates higher risk

target_customers['top_20_percent_risk'] = target_customers['risk_percentile'] <= 0.2



In [ ]:
# Now we have two strategies for targeting customers: uplift-based targeting and risk-based targeting.

# example: Customer A, Churn risk = 90%, Uplift = -2% (offer doesn't help, likely to churn)
# example: Customer B, Churn risk = 75%, Uplift = -30% (bit less risky, offer helps, likely to stay if offered)

#So uplift targeting can potentially spend the retention budget more efficiently.

In [ ]:
# Calculate expected incremental customers retained from our 182 uplift-targeted customers.
# For the customers we recommend targeting, calculate their average:
#Instead of targeting every high-risk customer, the uplift model prioritizes customers predicted to benefit most from the retention intervention.
uplift_cutoff = target_customers['uplift'].quantile(0.20)

target_customers['recommended_target'] = (
    target_customers['uplift'] <= uplift_cutoff
)

uplift_target = target_customers[
    target_customers['recommended_target']
].copy()

print("Customers targeted:", len(uplift_target))
print("Average uplift:", round(uplift_target['uplift'].mean(), 4))
print("Uplift cutoff:", round(uplift_cutoff, 4))

Customers targeted: 183
Average uplift: -0.3497
Uplift cutoff: -0.2586


In [ ]:
#Among the customers selected by the uplift model, the treatment is predicted to reduce churn probability by about 35 percentage points on average.
# [in other words] In the simulated uplift analysis, the recommended target group had an average estimated churn reduction of 34.97 percentage points under treatment.


In [ ]:
# Incremental revenue


absolute_uplift = -uplift_target['uplift'].mean()
incremental_customers_retained = absolute_uplift * len(uplift_target)
print(
    "Estimated incremental customers retained:",
    round(incremental_customers_retained, 1)
)
# 183 * 0.3497b= 64 

#The simulated uplift model estimates that targeting these 183 customers could retain approximately 64 additional customers compared with not providing the intervention.

Estimated incremental customers retained: 64.0


In [ ]:
#Among the customers selected by the uplift model, the treatment is predicted to reduce churn probability by about 35 percentage points on average.
# [in other words] In the simulated uplift analysis, the recommended target group had an average estimated churn reduction of 34.97 percentage points under treatment.


In [ ]:
# Incremental monthly revenue
avg_monthly_charge = uplift_target['MonthlyCharges'].mean()
monthly_incremental_revenue = (
    incremental_customers_retained * avg_monthly_charge
)
annua_incremental_revenue = monthly_incremental_revenue * 12


print(
    f"Estimated incremental monthly revenue: "
    f"${monthly_incremental_revenue:,.2f}",
    f"Estimated incremental annual revenue: "
    f"${annua_incremental_revenue:,.2f}"
)



Estimated incremental monthly revenue: $5,460.99 Estimated incremental annual revenue: $65,531.93


In [ ]:
#In the simulated uplift analysis, targeting the 183 customers with the strongest predicted treatment benefit was estimated to retain approximately 64 incremental customers, 
# generating approximately $65.5K in annual incremental revenue.

In [ ]:
# ROI
#suppose the cost of retentio offer is $50

incentive_cost_per_customer = 50
total_incentive_cost = len(uplift_target) * incentive_cost_per_customer
net_incremental_revenue = annua_incremental_revenue - total_incentive_cost  
roi = net_incremental_revenue/total_incentive_cost
print(
    f"Total incentive cost: ${total_incentive_cost:,.2f}",
    f"Net incremental revenue: ${net_incremental_revenue:,.2f}",
    f"ROI: {roi:.2f}"
)

Total incentive cost: $9,150.00 Net incremental revenue: $56,381.93 ROI: 6.16


In [ ]:
#ROI, Return on Investment: For every $1 spent on the retention intervention, the simulation estimates approximately $6.16 of net return.

# Predict
#   ↓
# Identify high-risk customers
#   ↓
# Explain risk with SHAP
#   ↓
# Target 911 high-risk month-to-month customers
#   ↓
# Uplift modeling
#   ↓
# Select 183 customers most likely to benefit
#   ↓
# ~64 incremental customers retained
#   ↓
# $65.5K annual incremental revenue
#   ↓
# $9.15K intervention cost
#   ↓
# $56.4K net incremental revenue
#   ↓
# 6.16 ROI

In [ ]:
#CUPED : Controlled Pre-experiment Data
# Make the A/B test's treatment-effect estimate more precise by using pre-experiment information.
# creating a simulated pre-experiment data for CUPED since in our dataset its not present (ex: previous-month churn risk, previous-month engagement,previous-month usage, previous-month spending, previous-period customer activity)
# Create a separate CUPED demo dataset
# why use cuped: "I would use CUPED in a randomized experiment when I have a pre-treatment metric that's correlated with the outcome. It reduces variance in the outcome and therefore gives me a more precise estimate of the treatment effect."
#CUPED uses a pre-treatment metric, such as previous churn risk, that is correlated with the post-treatment outcome."
cuped_demo = target_customers[['group']].copy()

np.random.seed(42)

# Simulated pre-experiment metric
cuped_demo['pre_period_metric'] = np.random.normal(
    100, 15, len(cuped_demo)
)

# Simulated post-experiment metric
# Treatment has a small positive effect
treatment_effect = np.where(
    cuped_demo['group'] == 'treatment',
    5,
    0
)

cuped_demo['post_period_metric'] = (
    cuped_demo['pre_period_metric'] * 0.7
    + np.random.normal(0, 10, len(cuped_demo))
    + treatment_effect
)

cuped_demo.head()



,group,pre_period_metric,post_period_metric
0,treatment,107.450712,82.407002
4,control,97.926035,71.042062
5,control,109.715328,92.575262
20,control,122.845448,85.038858
31,treatment,96.487699,75.331605


In [ ]:
# Now finding the correlation between the pre-experiment data and the outcome variable (churn probability) to calculate the CUPED adjustment coefficient.
correlation = cuped_demo[
    ['pre_period_metric', 'post_period_metric']
].corr()

print(correlation)

                    pre_period_metric  post_period_metric
pre_period_metric            1.000000            0.696664
post_period_metric           0.696664            1.000000


In [ ]:
#A correlation of 0.021 is extremely weak — essentially close to zero.
# That means your simulated pre_period_risk is not meaningfully correlated with the post-intervention churn outcome.
# A correlationn of 70% is perfectly reasonable, the pre-period metric and post-period metric have a strong positive relationship.
# So your simulated pre-period metric explains a meaningful amount of variation in the post-period metric.



In [ ]:
#Calculating theta
# theta definition: the CUPED adjustment coefficient, calculated as the covariance between the post-period metric and the pre-period metric divided by the variance of the pre-period metric.


theta = (
    cuped_demo['post_period_metric'].cov(
        cuped_demo['pre_period_metric']
    )
    / cuped_demo['pre_period_metric'].var()
)

print("Theta:", theta)
#Theta ≈ 0.68 means the pre-period metric is useful for explaining variation in the post-period metric.

Theta: 0.6821803823895037


In [ ]:
# removing predictable variance : theta * pre_period_metric - mean

cuped_demo['adjusted_post_period_metric'] = (
    cuped_demo['post_period_metric']
    - theta * (cuped_demo['pre_period_metric'] - cuped_demo['pre_period_metric'].mean())
)   
cuped_demo['adjusted_post_period_metric']

0       77.548404
4       72.681012
5       86.171788
20      69.678274
31      77.951760
          ...    
7008    62.669117
7010    73.312299
7018    65.524871
7032    74.489788
7041    70.910384
Name: adjusted_post_period_metric, Length: 911, dtype: float64

In [ ]:
print('original post-period metric:', cuped_demo['post_period_metric'].var())

print(cuped_demo['post_period_metric'])


print('adjusted post-period metric:', cuped_demo['adjusted_post_period_metric'].var())
print(cuped_demo['adjusted_post_period_metric'])

original post-period metric: 207.30786295231576
0       82.407002
4       71.042062
5       92.575262
20      85.038858
31      75.331605
          ...    
7008    48.529977
7010    80.724360
7018    71.905789
7032    96.401605
7041    67.536847
Name: post_period_metric, Length: 911, dtype: float64
adjusted post-period metric: 106.6929389090149
0       77.548404
4       72.681012
5       86.171788
20      69.678274
31      77.951760
          ...    
7008    62.669117
7010    73.312299
7018    65.524871
7032    74.489788
7041    70.910384
Name: adjusted_post_period_metric, Length: 911, dtype: float64


In [ ]:
#CUPED removed some predictable customer-to-customer variation using the pre-period metric, making the experiment estimate more precise.
# CUPED variance is smaller, that's the result we're looking for.



In [ ]:
# checking for treatment effect:
treatment_effect_original = (
    cuped_demo.loc[
        cuped_demo['group'] == 'treatment',
        'post_period_metric'
    ].mean()
    -
    cuped_demo.loc[
        cuped_demo['group'] == 'control',
        'post_period_metric'
    ].mean()
)

treatment_effect_cuped = (
    cuped_demo.loc[
        cuped_demo['group'] == 'treatment',
        'adjusted_post_period_metric'
    ].mean()
    -
    cuped_demo.loc[
        cuped_demo['group'] == 'control',
        'adjusted_post_period_metric'
    ].mean()
)

print("Original treatment effect:", treatment_effect_original)
print("CUPED treatment effect:", treatment_effect_cuped)

Original treatment effect: 4.688709794128911
CUPED treatment effect: 5.1224598371419745


In [ ]:
#The treatment effect is roughly similar before and after CUPED: 4.69 → 5.12
#So CUPED did not fundamentally change the estimated treatment effect.
#At the same time, your variance went: 207.31 → 106.69
#That's about a 48.5% reduction in variance.
# The treatment effect stayed approximately the same, while the variance decreased substantially. Therefore, the experiment became more precise.
# If I have a strong pre-experiment metric, such as prior customer engagement or spending, I can use CUPED to remove predictable variation and improve the statistical precision of the A/B test

In [ ]:
#CUPED doesn't directly reduce the cost of the intervention. Instead, it can reduce the sample size or experiment duration needed to detect a given effect, because the estimate is less noisy.

#Without CUPED: More noise → need more customers / longer experiment → potentially higher experiment cost.

# With CUPED

# Less noise → potentially need fewer customers / shorter experiment → potentially lower experiment cost.

# So for an interview, you can say:

# “CUPED reduces variance, which improves statistical power. That can allow us to detect an effect with fewer samples or in a shorter experiment, potentially reducing the cost and time of experimentation.”

# That's a stronger answer than simply saying “CUPED reduces cost.”

# And importantly, CUPED is not reducing the cost of giving the treatment to customers. It can reduce the cost of running the experiment.

## DID


In [ ]:
# CUPED: “Can I use pre-experiment information to make my A/B estimate more precise?”
# DiD: “Can I compare how treatment and control changed over time to estimate the treatment effect?”
# DiD → causal effect from changes over time
# Steps: Treatment vs. Control
# Before vs. After
# An outcome measured in both periods
# Compare the change in treatment against the change in control

# simulating before and after churn effect
import pandas as pd
import numpy as np

np.random.seed(42)

did_demo = target_customers[['group']].copy()

# Simulated churn rate before the intervention
did_demo['before_churn'] = np.random.normal(
    0.40, 0.05, len(did_demo)
)

# Simulated churn rate after the intervention
did_demo['after_churn'] = (
    did_demo['before_churn']
    - np.where(did_demo['group'] == 'treatment', 0.10, 0.02)
    + np.random.normal(0, 0.02, len(did_demo))
)

did_demo.head()

,group,before_churn,after_churn
0,treatment,0.424836,0.329219
4,control,0.393087,0.378074
5,control,0.432384,0.443933
20,control,0.476151,0.454246
31,treatment,0.388292,0.293873


In [27]:
# calculating change overtime for treatment and control group 
control_change = (
    did_demo.loc[
        did_demo['group'] == 'control',
        'after_churn'
    ].mean()
    -
    did_demo.loc[
        did_demo['group'] == 'control',
        'before_churn'
    ].mean()
)

treatment_change = (
    did_demo.loc[
        did_demo['group'] == 'treatment',
        'after_churn'
    ].mean()
    -
    did_demo.loc[
        did_demo['group'] == 'treatment',
        'before_churn'
    ].mean()
)

print("Control change:", control_change)
print("Treatment change:", treatment_change)

NameError: name 'did_demo' is not defined

In [ ]:
# Control Group: churn naturally decreased by about 1.88 pp even without treatment.
# Treatment Group: churn decresed among 10 pp due to the treatment.
# DiD = Treatment change − Control change

did_effect = treatment_change - control_change

print("Difference-in-Differences effect:", did_effect)

# Your DiD effect is:

#−0.07973 ≈ −7.97 percentage points means

#DiD relies on the parallel trends assumption:

#Without the intervention, the treatment and control groups would have followed similar trends over time.
# The treatment group experienced an additional 7.97 percentage-point reduction in churn relative to the control group, after accounting for the change that occurred in the control group over the same perio
# parallel trends assumption is crucial for the validity of the DiD estimate. that is the treatment and control groups would have followed similar trends in the absence of the treatment.


NameError: name 'treatment_change' is not defined

In [ ]:
# DiD vs A/B testing

# AB testing: Randomly give 50% of customers a retention offer and compare their churn with the 50% who don't receive it.

# DiD (Difference-in-Differences): Compare the churn of customers before and after a retention offer is given to one group, while another group does not receive the offer, controlling for time trends. 
# Use it when you have before-and-after data for treatment and comparison groups, especially when randomization isn't available.



In [ ]:
# You want to measure the impact of a new retention program. What method would you use?
#First I'd check whether we can randomly assign customers to treatment and control. If we can, I'd use an A/B test. 
#If randomization isn't possible but I have suitable treatment and comparison groups with pre- and post-period data, I would consider Difference-in-Differences, assuming the parallel-trends assumption is reasonable.
# With DiD, you don't necessarily create treatment and control by random assignment.

# Instead, the groups usually already exist because of how the real-world intervention happened.

# Example

# Suppose eBay introduces a new seller-support program only in California, but not yet in Texas.

# You could define:

# Treatment = California sellers
# Control   = Texas sellers






In [ ]:
# Example:
# Imagine eBay launches a new seller-support program.

# Before the program,	Churn control	40%
# Treatment	45%
# After the program
# Group	Churn
# Control	38%
# Treatment	30%

# Now calculate:

# Control change

# 38% − 40% = -2 pp

# Treatment change

# 30% − 45% = -15 pp


# DID: -15 pp - (-2 pp)
# = -13 pp (The estimated treatment effect is a 13 percentage-point reduction in churn, relative to the control group's change.)

In [ ]:
 #How would you determine the sample size for an A/B test?”, you can say:

# “I would use power analysis based on the baseline metric, minimum detectable effect, significance level, and desired statistical power. For example, I might use a 5% significance level and 80% power, 
# then calculate the sample size needed to detect the minimum effect that would be meaningful to the business.”

# Is the DiD effect statistically significant? 
# DID Effect

In [ ]:
Question: Is this difference likely to represent a real treatment effect, or could it just be random variation?
Null hypothesis (H₀): The treatment has no additional effect compared with control.

In DiD terms:
H₀: DiD = 0

Alternative hypothesis (H₁): The treatment has an effect.
H₁: DiD != 0

# For your earlier A/B test, you used a two-proportion z-test because the outcome was binary churn.
# For DiD, we generally use a regression model with a treatment × post interaction to estimate and test the DiD effect.
# regression model with a treatment × post interaction to estimate and test the DiD effect.
WIP
